In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import os

import anndata as ad
import colorcet as cc
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scvi
import seaborn as sns
from rich import print

In [4]:
from dataclasses import dataclass
from typing import Optional


@dataclass(frozen=True)
class PARAMS:
    """set all params for our analysis"""

    # DATA_FOLDER: str = (
    #     "/home/labs/nyosef/Collaboration/SpatialDatasets/xenium/human_breast_cancer/"
    # )
    DATA_FOLDER: str = "/home/nathanlevy/sda/Data/"
    DATA_FILE: str = "proseg_resolvi_scanvi_xenium_breast_cancer_S1_R1_2.h5ad"

    FIGURES_FOLDER: str = "figures/"

    # for the data
    COORDS: str = "spatial"
    BATCH: str = "sample"
    SAMPLE: str = "sample"
    CELL_TYPE: str = "predictions_resolvi_proseg_coarse_corrected"
    NICHE: str = "kmeans_alpha_5"

    # for scVI
    EXPRESSION_MODEL = "scanvi"
    N_LAYERS: int = 1
    N_LATENT: int = 10
    LIKELIHOOD: str = "poisson"
    ######################
    N_EPOCHS_SCVI: int = 1001
    N_EPOCHS_RESOLVI: int = 100
    LR_SCVI: float = 5e-4
    BATCH_SIZE_SCVI: int = 512
    OPTIMIZER: str = "Adam"
    ######################
    WEIGHT_DECAY: float = 1e-6
    KL_WARMUP: Optional[int] = 400
    MAX_KL_WEIGHT: float = 1

    # for nicheVI
    K_NN: int = 20  # TODO try also less cells per niche, like 12
    N_LAYERS_NICHE: int = 1
    N_LAYERS_COMPO: int = 1
    N_HIDDEN: int = 128
    N_HIDDEN_COMPO: int = 128  # TODO make it adaptable
    N_HIDDEN_NICHE: int = 128
    N_LATENT_NICHEVI: int = 10
    ######################
    N_EPOCHS_NICHEVI: int = 1000
    LR_NICHEVI: float = 1e-3
    BATCH_SIZE_NICHEVI: int = 512
    REDUCE_LR_ON_PLATEAU: bool = True
    SPATIAL_WARMUP: int | None = None
    MIN_SPATIAL_WEIGHT: float = 1.0 if SPATIAL_WARMUP is None else 0
    MAX_SPATIAL_WEIGHT: float = 1.0
    USE_LAYER_NORM: bool = True
    USE_BATCH_NORM: bool = False


setup = PARAMS()

In [5]:
sc.set_figure_params(dpi=150, fontsize=12, format="svg")
figure_dir = setup.FIGURES_FOLDER

plots = True

In [6]:
scvi.settings.seed = 34

Global seed set to 34


In [7]:
data_dir, data_file = setup.DATA_FOLDER, setup.DATA_FILE

data_file_name = os.path.splitext(data_file)[0]
print(data_file_name)

path_to_save = os.path.join("../niche-VI-experiments/checkpoints", data_file_name)
os.makedirs(path_to_save, exist_ok=True)

proseg_resolvi_scanvi_xenium_breast_cancer_S1_R1_2

In [8]:
# adata = ad.read_h5ad(data_dir + data_file)
adata = ad.read_h5ad(path_to_save + "/proseg_resolvi_scanvi_xenium_breast_cancer_S1_R1_2_nicheVI.h5ad")

del adata.uns["log1p"]

print(adata)

AnnData object with n_obs × n_vars = 278658 × 313
    obs: 'sample', '_scvi_batch', 'cell_type', '_scvi_labels', 'batch', 'predictions_scanvi', 'x', 'y', 'z', 
'observed_x', 'observed_y', 'observed_z', 'fov', 'probability', 'background', 'confusion', 'n_counts', '_indices', 
'_scvi_ind_x', 'predicted_celltype_resolvi', 'predicted_celltype_prob_resolvi', 'predictions_resolvi_proseg', 
'predictions_resolvi_proseg_coarse', 'predicted_celltype_resolvi_coarse', 'index', 'kmeans_alpha_4', 
'kmeans_alpha_5', 'kmeans_alpha_5_transferred', 'predictions_resolvi_proseg_coarse_corrected', '_scvi_sample'
    var: 'mt'
    uns: '_scvi_manager_uuid', '_scvi_uuid', 'pca'
    obsm: 'X_banksy_02_harmony', 'X_pca', 'X_resolvi', 'X_resolvi_MDE', 'X_resolvi_proseg', 'X_scANVI', 
'X_scANVI_MDE', 'X_scanvi', 'X_spatial', 'banksy_pc_20_lambda_0.0', 'banksy_pc_20_lambda_0.2', 
'celltype_transfer_resolvi', 'distance_neighbor', 'index_neighbor', 
'ln_s20_h64_w400_scanvi_lr0.001_poisson_X_nicheVI', 'ln_s20_w400_scanvi_lr0.001_poisson_X_nicheVI', 
'mog_ln_s20_h64_w400_scanvi_lr0.001_poisson_X_nicheVI', 'mog_ln_s20_w400_scanvi_lr0.001_poisson_X_nicheVI', 
'neighborhood_composition', 'niche_composition', 'niche_distances', 'niche_indexes', 'observed_spatial', 'qz1_m', 
'qz1_m_niche_ct', 'spatial'
    varm: 'PCs'
    layers: 'counts', 'counts_wo_bg'

In [9]:
# adata_CD8 = adata[adata.obs[setup.CELL_TYPE].isin(["CD8+_T_Cells"])].copy()
# adata_CD8.obsm["MDE_CD8+_T_Cells_nichevi"] = scvi.model.utils.mde(
#     adata_CD8.obsm[model+"_X_nicheVI"]
# )
# adata_CD8.obsm["MDE_CD8+_T_Cells_scvi"] = scvi.model.utils.mde(adata_CD8.obsm[KEY_1])

adata_CD8 = ad.read_h5ad(path_to_save + "/xenium_breast_cancer_S1_R1_2_nicheVI_CD8.h5ad").copy()

del adata_CD8.uns["log1p"]

print(adata_CD8)

AnnData object with n_obs × n_vars = 16054 × 313
    obs: 'sample', '_scvi_batch', 'cell_type', '_scvi_labels', 'batch', 'predictions_scanvi', 'x', 'y', 'z', 
'observed_x', 'observed_y', 'observed_z', 'fov', 'probability', 'background', 'confusion', 'n_counts', '_indices', 
'_scvi_ind_x', 'predicted_celltype_resolvi', 'predicted_celltype_prob_resolvi', 'predictions_resolvi_proseg', 
'predictions_resolvi_proseg_coarse', 'predicted_celltype_resolvi_coarse', 'index', 'kmeans_alpha_4', 
'kmeans_alpha_5', 'kmeans_alpha_5_transferred', 'predictions_resolvi_proseg_coarse_corrected', '_scvi_sample', 
'CD8+_T_Cells_to_DCIS', 'CD8+_T_Cells_to_invasive', 'leiden_nichevi', 'leiden_scvi', 'leiden_banksy_02'
    var: 'mt'
    uns: '_scvi_manager_uuid', '_scvi_uuid', 'kmeans_alpha_5_colors', 'leiden', 'leiden_nichevi_colors', 
'neighbors', 'pca', 'predictions_resolvi_proseg_coarse_corrected_colors', 'sample_colors', 'umap'
    obsm: 'X_banksy_02_harmony', 'X_pca', 'X_resolvi', 'X_resolvi_MDE', 'X_resolvi_proseg', 'X_scANVI', 
'X_scANVI_MDE', 'X_scanvi', 'X_scanvi_MDE', 'X_spatial', 'X_umap', 'banksy_pc_20_lambda_0.0', 
'banksy_pc_20_lambda_0.2', 'celltype_transfer_resolvi', 'distance_neighbor', 'index_neighbor', 
'ln_s20_h64_w400_scanvi_lr0.001_poisson_MDE', 'ln_s20_h64_w400_scanvi_lr0.001_poisson_X_nicheVI', 
'ln_s20_w400_scanvi_lr0.001_poisson_MDE', 'ln_s20_w400_scanvi_lr0.001_poisson_X_nicheVI', 
'mog_ln_s20_h64_w400_scanvi_lr0.001_poisson_MDE', 'mog_ln_s20_h64_w400_scanvi_lr0.001_poisson_X_nicheVI', 
'mog_ln_s20_w400_scanvi_lr0.001_poisson_MDE', 'mog_ln_s20_w400_scanvi_lr0.001_poisson_X_nicheVI', 
'neighborhood_composition', 'niche_composition', 'niche_distances', 'niche_indexes', 'observed_spatial', 'qz1_m', 
'qz1_m_niche_ct', 'spatial'
    varm: 'PCs'
    layers: 'counts', 'counts_wo_bg'
    obsp: 'connectivities', 'distances'

In [10]:
# for sample in adata.obs[setup.SAMPLE].unique().tolist()[:]:
#     # with plt.rc_context():
#     sc.pl.spatial(
#         adata_CD8[adata_CD8.obs[setup.SAMPLE] == sample],
#         spot_size=80,
#         # color=["leiden_nichevi"],
#         color=[f"{setup.NICHE}_transferred", "leiden_nichevi"],
#         # groups=[
#         #     "0",
#         #     "2",
#         # ],
#         # groups=["Stromal", "Endothelial", "Macrophages_1"],
#         ncols=1,
#         frameon=False,
#         title=sample,
#         # cmap="Set2",
#         palette=cc.glasbey_dark,
#         show=False,
#     )

In [11]:
if setup.EXPRESSION_MODEL == "scanvi":
    scvivae = scvi.model.SCANVI.load(
        dir_path=path_to_save
        + f"/{setup.EXPRESSION_MODEL}vae_E"
        + str(setup.N_EPOCHS_SCVI)
        + "_"
        + str(setup.LIKELIHOOD)
        + ".pt",
        adata=adata,
        accelerator="cpu",
    )
if setup.EXPRESSION_MODEL == "scvi":
    scvivae = scvi.model.SCVI.load(
        dir_path=path_to_save
        + f"/{setup.EXPRESSION_MODEL}vae_E"
        + str(setup.N_EPOCHS_SCVI)
        + "_"
        + str(setup.LIKELIHOOD)
        + ".pt",
        adata=adata,
        accelerator="cpu",
    )

INFO     File                                                                                                      
         ../niche-VI-experiments/checkpoints/proseg_resolvi_scanvi_xenium_breast_cancer_S1_R1_2/scanvivae_E1001_poi
         sson.pt/model.pt already downloaded                                                                       


In [12]:
MODEL = "ln_s20_h64_w400_scanvi_lr0.001_poisson"

DELTA = 0.05
CORRUPTION = None
FDR = 0.05

adata.obs["leiden_nichevi"] = "Unknown"
adata.obs.loc[adata.obs[setup.CELL_TYPE] == "Endothelial", "leiden_nichevi"] = adata_CD8.obs["leiden_nichevi"]

In [13]:
adata.obs["leiden_nichevi"].value_counts()

leiden_nichevi
Unknown    262604
0            3522
1            3425
2            2809
3            1889
4            1869
5            1604
6             936
Name: count, dtype: int64

In [14]:
import nichevi
from nichevi.differential_expression import _fdr_de_prediction

model = MODEL

path_to_save_nichevae = path_to_save + "/nichevae_" + model + "_" + str(setup.N_EPOCHS_NICHEVI) + ".pt"

nichevae = nichevi.nicheSCVI.load(
    dir_path=path_to_save_nichevae,
    adata=adata,
    accelerator="cpu",
)

INFO     File                                                                                                      
         ../niche-VI-experiments/checkpoints/proseg_resolvi_scanvi_xenium_breast_cancer_S1_R1_2/nichevae_ln_s20_h64
         _w400_scanvi_lr0.001_poisson_1000.pt/model.pt already downloaded                                          


In [15]:
scvi_DE = scvivae.differential_expression(
    adata,
    groupby="leiden_nichevi",
    group1="0",
    group2="2",
    batch_correction=False,
    delta=DELTA,
)

scvi_DE

DE...:   0%|          | 0/1 [00:00<?, ?it/s]

DE...: 100%|██████████| 1/1 [00:00<00:00,  4.13it/s]


,proba_de,proba_not_de,bayes_factor,scale1,scale2,pseudocounts,delta,lfc_mean,lfc_median,lfc_std,...,raw_mean1,raw_mean2,non_zeros_proportion1,non_zeros_proportion2,raw_normalized_mean1,raw_normalized_mean2,is_de_fdr_0.05,comparison,group1,group2
EIF4EBP1,0.7936,0.2064,1.570634,0.001682,0.001178,0.0,0.05,0.503431,0.517210,0.568281,...,0.440375,0.200427,0.269733,0.149163,17.970189,10.425084,False,0 vs 2,0,2
AQP1,0.7874,0.2126,-1.449462,0.036358,0.063374,0.0,0.05,-0.956908,-0.902339,1.153154,...,7.398637,12.027056,0.865133,0.973300,335.107652,678.485662,False,0 vs 2,0,2
CENPF,0.7846,0.2154,1.360875,0.001625,0.000547,0.0,0.05,1.506073,1.504177,1.900154,...,0.419364,0.108936,0.135718,0.055536,16.087065,5.103917,False,0 vs 2,0,2
TOP2A,0.7794,0.2206,1.333565,0.003272,0.001040,0.0,0.05,1.632946,1.642280,2.106929,...,0.876491,0.187611,0.177740,0.060520,31.021872,8.804542,False,0 vs 2,0,2
SQLE,0.7722,0.2278,1.456287,0.002659,0.001893,0.0,0.05,0.482089,0.497821,0.594141,...,0.675468,0.352439,0.339864,0.233535,26.687824,18.937746,False,0 vs 2,0,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
POLR2J3,0.4732,0.5268,0.082794,0.018304,0.017989,0.0,0.05,0.019153,0.021974,0.427637,...,4.298978,3.351727,0.895514,0.838733,185.278321,179.848987,False,0 vs 2,0,2
TOMM7,0.4680,0.5320,-0.107262,0.014003,0.014302,0.0,0.05,-0.025429,-0.022112,0.354262,...,3.258944,2.740477,0.811755,0.783909,135.626098,145.035787,False,0 vs 2,0,2
PDE4A,0.4462,0.5538,0.014447,0.001641,0.001639,0.0,0.05,0.001637,0.002619,0.353201,...,0.369108,0.326807,0.273992,0.241011,16.361553,18.321033,False,0 vs 2,0,2
LARS,0.4452,0.5548,0.096614,0.005560,0.005503,0.0,0.05,0.012048,0.017903,0.267440,...,1.219478,1.159131,0.570698,0.540762,52.011745,61.436791,False,0 vs 2,0,2


In [16]:
example_DE = nichevae.differential_expression(
    adata,
    # groupby="kmeans_alpha_5",
    groupby="leiden_nichevi",
    # groupby=setup.CELL_TYPE,
    group1="0",
    group2="2",
    batch_correction=False,
    radius=None,
    k_nn=6,
    delta=DELTA,
    count_corruption=CORRUPTION,
    niche_mode=True,
    n_samples=100,
    fdr_target=FDR,
    # weights="importance",
    pseudocounts=0.0,
)

Computing nearest neighbors...
(278658, 2)
(278658, 1)
(278658, 1)


: 

In [ ]:
df_g1_g2 = example_DE["group1_group2"]
df_n1_g2 = example_DE["corrupted1_group2"] if CORRUPTION is not None else example_DE["niche1_group2"]
df_g1_n1 = example_DE["group1_corrupted1"] if CORRUPTION is not None else example_DE["group1_niche1"]